# Day 36 — **Redis: Agent Shared Memory Store**

**Phase 2 · Module 5: Multi-Agent Orchestration** · ~30 min

Multiple agents can't share state through Python variables — they run in different tasks, processes, maybe different machines. **Redis** is the standard external store for agent working memory: fast, TTL-native, with pub/sub for agent-to-agent signalling. It's the real backing for the `SharedMemory` mock from today's module notebook.

No Redis server in a notebook, so below is a **`MiniRedis`** with the exact `redis-py` method names (`hset/hget/expire/setex/publish/subscribe`). Swap `MiniRedis()` → `redis.Redis(decode_responses=True)` and the code is unchanged. Install for real: `pip install redis`.

Today:
1. **Hashes** — store an agent's task results by session, read them back.
2. **TTL** — `expire`/`setex` so agent sessions self-clean.
3. **Pub/Sub** — one agent publishes a result, another reacts.
4. **JSON values** — the serialise/deserialise discipline Redis forces on you.

## 1. Hashes — results keyed by session

`hset(name, key, value)` stores a field in a hash; `hget`/`hgetall` read it back. Use the `session_id` as the hash name so each workflow's data is namespaced and isolated.

In [1]:
import time, json, fnmatch

class MiniRedis:
    """In-notebook stand-in for redis-py (decode_responses=True). Same method names."""
    def __init__(self):
        self._h: dict = {}         # hashes
        self._kv: dict = {}        # strings
        self._exp: dict = {}       # key -> expiry epoch
        self._subs: dict = {}      # channel -> list[callback]

    def _expired(self, key):
        e = self._exp.get(key)
        if e and time.time() > e:
            self._h.pop(key, None); self._kv.pop(key, None); self._exp.pop(key, None)
            return True
        return False

    def hset(self, name, key=None, value=None, mapping=None):
        self._h.setdefault(name, {})
        if mapping: self._h[name].update({k: str(v) for k, v in mapping.items()})
        if key is not None: self._h[name][key] = str(value)

    def hget(self, name, key):
        return None if self._expired(name) else self._h.get(name, {}).get(key)

    def hgetall(self, name):
        return {} if self._expired(name) else dict(self._h.get(name, {}))

    def expire(self, name, ttl): self._exp[name] = time.time() + ttl
    def ttl(self, name): return int(self._exp.get(name, time.time()) - time.time())
    def setex(self, key, ttl, value): self._kv[key] = str(value); self.expire(key, ttl)
    def get(self, key): return None if self._expired(key) else self._kv.get(key)
    def keys(self, pattern="*"):
        return [k for k in {**self._h, **self._kv} if fnmatch.fnmatch(k, pattern)]

    def publish(self, channel, message):
        for cb in self._subs.get(channel, []): cb(message)
        return len(self._subs.get(channel, []))
    def subscribe(self, channel, callback): self._subs.setdefault(channel, []).append(callback)

r = MiniRedis()                                        # real: redis.Redis(decode_responses=True)
r.hset("sess:77", mapping={"enrich": "done", "score": "BBB"})
r.hset("sess:77", "limit", "750000")
print("one field :", r.hget("sess:77", "score"))
print("all fields:", r.hgetall("sess:77"))

one field : BBB
all fields: {'enrich': 'done', 'score': 'BBB', 'limit': '750000'}


☝ Namespacing by `sess:77` is the isolation boundary — colon-delimited key prefixes are the Redis convention and let you `keys("sess:*")` to sweep a workflow. Note everything is coerced to `str`: Redis stores bytes/strings only, which is why real `redis-py` is used with `decode_responses=True` and why structured values need JSON (cell 4). `hset(..., mapping=...)` writes several fields atomically — one round-trip instead of N.

## 2. TTL — self-cleaning sessions

Agent working-memory must expire or Redis fills with dead sessions. `expire(key, seconds)` sets a lifetime; `setex(key, ttl, value)` writes-with-TTL in one call. A 24h TTL is typical for a workflow's scratch state.

In [2]:
r.expire("sess:77", 24 * 3600)
print("ttl on sess:77:", r.ttl("sess:77"), "seconds")

# setex: write a cache entry that dies in 1 second
r.setex("quote:GBPUSD", 1, json.dumps({"rate": 1.27}))
print("fresh cache:", r.get("quote:GBPUSD"))

r._exp["quote:GBPUSD"] = time.time() - 1               # force-expire for the demo
print("after expiry:", r.get("quote:GBPUSD"))          # None -> auto-removed

ttl on sess:77: 86399 seconds
fresh cache: {"rate": 1.27}
after expiry: None


☝ TTL is the difference between a store that stays healthy and one that OOMs in a week. Two idioms: `expire` on an existing session to bound its lifetime, and `setex` for cache entries that must be fresh (an FX quote good for seconds). When the key expires, the next read returns `None` — so callers must handle a cache **miss**, which is exactly the fallback-to-source path you want anyway. Never store agent state without a TTL unless it's *meant* to be durable (then it belongs in DynamoDB/Postgres, not Redis scratch).

## 3. Pub/Sub — agent-to-agent signalling

Beyond shared state, agents **notify** each other: a scoring agent finishes and publishes; a reporting agent subscribed to that channel reacts. Redis pub/sub decouples them — the publisher doesn't know who's listening.

In [3]:
received = []

def report_agent_listener(message: str):
    event = json.loads(message)
    received.append(event)
    print(f"  [report-agent] woke on: {event['type']} for {event['customer']}")

# report agent subscribes to the 'scoring.done' channel
r.subscribe("scoring.done", report_agent_listener)

# scoring agent finishes and publishes — fire-and-forget
n = r.publish("scoring.done", json.dumps({"type": "scored", "customer": "C-4417", "grade": "BBB"}))
print(f"published to {n} subscriber(s); received {len(received)} event(s)")

  [report-agent] woke on: scored for C-4417
published to 1 subscriber(s); received 1 event(s)


☝ Pub/sub is **fire-and-forget decoupling**: the scoring agent calls `publish` and moves on, unaware of subscribers — you can add a fraud-monitor listener later with zero change to the publisher. The trade-off to know for an interview: Redis pub/sub has **no delivery guarantee** (a subscriber that's down misses the message). For work that *must* be processed, use a durable queue (Redis Streams, SQS, Kafka — Day 33's queue pattern), not pub/sub. Pub/sub is for signals; streams are for tasks.

## 4. JSON values — serialise on the boundary

Redis stores strings. A dict must be `json.dumps`'d on write and `json.loads`'d on read. Wrap that so agents never touch raw Redis and can't forget a step.

In [4]:
class AgentStore:
    """Typed façade over Redis: agents store/get dicts, JSON handled here."""
    def __init__(self, backend, ttl=24 * 3600):
        self._r, self._ttl = backend, ttl
    def put(self, session: str, key: str, value: dict) -> None:
        self._r.hset(session, key, json.dumps(value))
        self._r.expire(session, self._ttl)
    def get(self, session: str, key: str) -> dict | None:
        raw = self._r.hget(session, key)
        return json.loads(raw) if raw is not None else None

store = AgentStore(r)
store.put("sess:88", "analysis", {"grade": "AAA", "pd": 0.011, "limit": 500_000})
got = store.get("sess:88", "analysis")
print("round-tripped dict:", got, "| pd type:", type(got["pd"]).__name__)

round-tripped dict: {'grade': 'AAA', 'pd': 0.011, 'limit': 500000} | pd type: float


☝ Centralising `json.dumps`/`loads` in one façade means an agent stores a **dict** and gets a **dict** back — no stray strings, no forgotten deserialise, and the TTL is applied on every write automatically. This is the same "hide the integration behind an interface" move as Day 34's transport adapter: swap `MiniRedis` for `redis.Redis` (or `redis.asyncio.Redis` for the async agents of Day 35) and every agent using `AgentStore` keeps working untouched.

## Recap

| Need | redis-py | Notebook |
|---|---|---|
| Share results, namespaced | `hset/hget/hgetall` on `sess:*` | 1 |
| Self-clean sessions | `expire` / `setex` (TTL) | 2 |
| Signal between agents | `publish` / `subscribe` | 3 |
| Store structured data | `json.dumps`/`loads` in a façade | 4 |
| Durable task delivery | Streams/SQS — **not** pub/sub | 3 |

**Swap to prod:** `MiniRedis()` → `redis.Redis(decode_responses=True)`; nothing else changes. Tomorrow (Day 37): `tenacity` — retry and circuit-breaker decorators for the tool calls that hit these stores and services.